# STS-B cosine-threshold sweep

This notebook loads the GLUE STS-B validation split from Hugging Face, binarizes human similarity scores using `score >= 4`, embeds both sentences with the configured FastEmbed model, and reports measured cosine-only precision/recall/FPR across thresholds. It reports cosine computation latency separately from the embedding batch. No benchmark results are checked into this notebook; run the cells to obtain results for your environment.

In [ ]:
import numpy as np
from datasets import load_dataset

from aegis.engine.embeddings import FastEmbedder
from benchmarks.evaluation import (
    PairExample,
    score_pair_cosines,
    stsb_binary_labels,
    threshold_sweep,
)

In [ ]:
dataset = load_dataset("nyu-mll/glue", "stsb", split="validation")
human_scores = [float(row["label"]) for row in dataset]
labels = stsb_binary_labels(human_scores)
examples = [
    PairExample(
        candidate=str(row["sentence1"]),
        query=str(row["sentence2"]),
        label=label,
    )
    for row, label in zip(dataset, labels, strict=True)
]
len(examples), sum(labels)

In [ ]:
embedder = FastEmbedder()
try:
    pair_scores, embedding_seconds, cosine_seconds = await score_pair_cosines(
        examples,
        embedder,
    )
finally:
    await embedder.close()

similarities = [score.cosine_similarity for score in pair_scores]
thresholds = np.linspace(-1.0, 1.0, 101).tolist()
results = threshold_sweep(labels, similarities, thresholds)
print(f"Embedding batch seconds: {embedding_seconds:.6f}")
print(
    f"Cosine scoring mean milliseconds per pair: {np.mean(cosine_seconds) * 1000:.6f}"
)

In [ ]:
display(results)